# nn-chain-explorer — Analysis Notebook

This notebook analyzes the topology of learned image embedding spaces by tracing nearest-neighbor chains.

**Prerequisites:** Run `python main.py run-all` before executing this notebook.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from scipy import stats as scipy_stats

try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

sns.set_theme(style='darkgrid', palette='husl')
plt.rcParams['figure.dpi'] = 100

ROOT = Path('..').resolve()
EMBEDDINGS_DIR = ROOT / 'embeddings'
RESULTS_DIR = ROOT / 'results'
DATA_DIR = ROOT / 'data'

def load_json(path, label=''):
    try:
        with open(path) as f:
            return json.load(f)
    except FileNotFoundError:
        print(f'[SKIP] {path} not found. Run: python main.py {label}')
        return None

def load_npy(path, label=''):
    try:
        return np.load(path)
    except FileNotFoundError:
        print(f'[SKIP] {path} not found. Run: python main.py {label}')
        return None

# Load all results
image_index = load_json(EMBEDDINGS_DIR / 'index.json', 'embed --model all')
chains_d = load_json(RESULTS_DIR / 'chains_dinov2.json', 'trace --model dinov2')
chains_c = load_json(RESULTS_DIR / 'chains_clip.json', 'trace --model clip')
stats_d = load_json(RESULTS_DIR / 'stats_dinov2.json', 'analyze --model dinov2')
stats_c = load_json(RESULTS_DIR / 'stats_clip.json', 'analyze --model clip')
comparison = load_json(RESULTS_DIR / 'comparison.json', 'analyze --compare')
emb_d = load_npy(EMBEDDINGS_DIR / 'dinov2.npy', 'embed --model dinov2')
emb_c = load_npy(EMBEDDINGS_DIR / 'clip.npy', 'embed --model clip')
nn_d = load_npy(RESULTS_DIR / 'nn_map_dinov2.npy', 'trace --model dinov2')
nn_c = load_npy(RESULTS_DIR / 'nn_map_clip.npy', 'trace --model clip')

print('All results loaded.' if all(x is not None for x in [image_index, chains_d, chains_c]) else 'Some results missing — run the pipeline first.')

## Cell 2 — Dataset Overview

In [ ]:
if image_index is None:
    print('[SKIP] image_index not available.')
else:
    from collections import Counter
    N = len(image_index)
    class_counts = Counter(entry['class'] for entry in image_index)
    print(f'Total images: {N}')
    print(f'Classes: {sorted(class_counts.keys())}')

    fig, ax = plt.subplots(figsize=(10, 4))
    classes = sorted(class_counts.keys())
    counts = [class_counts[c] for c in classes]
    bars = ax.bar(classes, counts, color=sns.color_palette('husl', len(classes)))
    ax.bar_label(bars)
    ax.set_title('Dataset — Images per Class', fontsize=14)
    ax.set_xlabel('Class')
    ax.set_ylabel('Count')
    plt.tight_layout()
    plt.show()
    print(f'Class distribution: {dict(class_counts)}')

## Cell 3 — Example Chains (Image Strips)

In [ ]:
from PIL import Image

if image_index is None or chains_d is None:
    print('[SKIP] Required data not available.')
else:
    rng = np.random.default_rng(42)
    sample_ids = rng.choice(len(image_index), size=min(10, len(image_index)), replace=False)

    def load_thumb(path, size=64):
        try:
            img = Image.open(path).convert('RGB').resize((size, size))
            return np.array(img)
        except Exception:
            return np.full((size, size, 3), 128, dtype=np.uint8)

    def render_chain_strip(ax, chain_data, image_index, max_show=8):
        chain = chain_data['chain']
        cycle_entry = chain_data['cycle_entry_node']
        if len(chain) > max_show:
            chain = chain[:max_show - 1] + [chain[-1]]
        imgs = []
        for nid in chain:
            if nid >= 0 and nid < len(image_index):
                imgs.append(load_thumb(image_index[nid]['path']))
        if not imgs:
            return
        strip = np.hstack(imgs)
        ax.imshow(strip)
        ax.axis('off')
        # Mark cycle entry
        for j, nid in enumerate(chain):
            if nid == cycle_entry:
                ax.add_patch(plt.Rectangle((j * 64 - 2, -2), 68, 68, fill=False, edgecolor='gold', linewidth=2))

    fig, axes = plt.subplots(len(sample_ids), 2, figsize=(14, len(sample_ids) * 1.5))
    if len(sample_ids) == 1:
        axes = [axes]

    for row, sid in enumerate(sample_ids):
        sid = int(sid)
        for col, (chains_data, label) in enumerate([(chains_d, 'DINOv2'), (chains_c, 'CLIP')]):
            if chains_data is None:
                axes[row][col].axis('off')
                continue
            c = chains_data['chains'][str(sid)]
            ax = axes[row][col]
            render_chain_strip(ax, c, image_index)
            ax.set_ylabel(f'{label}\nτ={c["transient_length"]} ℓ={c["cycle_length"]}', fontsize=8, rotation=0, labelpad=60)

    fig.suptitle('Example NN Chains (gold border = cycle entry)', fontsize=13)
    plt.tight_layout()
    plt.show()

## Cell 4 — Transient Length Distributions

In [ ]:
if chains_d is None and chains_c is None:
    print('[SKIP] No chain data available.')
else:
    fig, ax = plt.subplots(figsize=(10, 5))
    for chains_data, label, color in [
        (chains_d, 'DINOv2', '#4a6fa5'),
        (chains_c, 'CLIP', '#e07040'),
    ]:
        if chains_data is None:
            continue
        tau_vals = [chains_data['chains'][str(i)]['transient_length']
                    for i in range(chains_data['metadata']['n_images'])]
        tau_arr = np.array(tau_vals)
        ax.hist(tau_arr, bins=range(int(tau_arr.max()) + 2), alpha=0.5, label=label, color=color, density=True)
        try:
            sns.kdeplot(tau_arr, ax=ax, color=color, linewidth=2)
        except Exception:
            pass
        mean, std = tau_arr.mean(), tau_arr.std()
        ax.axvline(mean, color=color, linestyle='--', alpha=0.8)
        ax.annotate(f'{label}\nμ={mean:.2f}±{std:.2f}',
                    xy=(mean, ax.get_ylim()[1] * 0.5),
                    xytext=(mean + 0.5, ax.get_ylim()[1] * 0.5),
                    fontsize=9, color=color)

    ax.set_xlabel('Transient length τ')
    ax.set_ylabel('Density')
    ax.set_title('Transient Length Distributions — DINOv2 vs CLIP')
    ax.legend()
    plt.tight_layout()
    plt.show()

## Cell 5 — Cycle Length Distributions

In [ ]:
if chains_d is None and chains_c is None:
    print('[SKIP] No chain data available.')
else:
    fig, ax = plt.subplots(figsize=(10, 5))
    for chains_data, label, color in [
        (chains_d, 'DINOv2', '#4a6fa5'),
        (chains_c, 'CLIP', '#e07040'),
    ]:
        if chains_data is None:
            continue
        cyc_vals = [chains_data['chains'][str(i)]['cycle_length']
                    for i in range(chains_data['metadata']['n_images'])
                    if chains_data['chains'][str(i)]['terminated_by'] == 'cycle']
        if not cyc_vals:
            continue
        cyc_arr = np.array(cyc_vals)
        ax.hist(cyc_arr, bins=range(int(cyc_arr.max()) + 2), alpha=0.5, label=label, color=color, density=True)
        mean, std = cyc_arr.mean(), cyc_arr.std()
        ax.axvline(mean, color=color, linestyle='--', alpha=0.8)
        ax.annotate(f'{label}\nμ={mean:.2f}±{std:.2f}',
                    xy=(mean, ax.get_ylim()[1] * 0.6),
                    xytext=(mean + 0.1, ax.get_ylim()[1] * 0.6),
                    fontsize=9, color=color)

    ax.set_xlabel('Cycle length ℓ')
    ax.set_ylabel('Density')
    ax.set_title('Cycle Length Distributions — DINOv2 vs CLIP')
    ax.legend()
    plt.tight_layout()
    plt.show()

## Cell 6 — Hub In-degree Distributions (log-log)

In [ ]:
if nn_d is None and nn_c is None:
    print('[SKIP] No nn_map data available.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, nn_map, label, color in [
        (axes[0], nn_d, 'DINOv2', '#4a6fa5'),
        (axes[1], nn_c, 'CLIP', '#e07040'),
    ]:
        if nn_map is None:
            ax.set_visible(False)
            continue
        N = len(nn_map)
        in_degree = np.bincount(nn_map, minlength=N)
        # Log-log scatter: each image as one dot
        nonzero = in_degree[in_degree > 0]
        # Rank vs degree
        sorted_deg = np.sort(nonzero)[::-1]
        ranks = np.arange(1, len(sorted_deg) + 1)
        ax.scatter(ranks, sorted_deg, s=6, alpha=0.6, color=color)
        ax.set_xscale('log')
        ax.set_yscale('log')
        ax.set_xlabel('Rank')
        ax.set_ylabel('In-degree')
        ax.set_title(f'{label} — Hub In-degree (log-log)')
        ax.annotate(f'Max hub: {sorted_deg[0]}\nMedian: {np.median(nonzero):.1f}',
                    xy=(0.65, 0.85), xycoords='axes fraction', fontsize=9)

    plt.tight_layout()
    plt.show()

## Cell 7 — Hub Gallery (Top-20)

In [ ]:
from PIL import Image

def show_hub_gallery(nn_map, image_index, label, top_n=20):
    if nn_map is None or image_index is None:
        print(f'[SKIP] {label}: missing data.')
        return
    N = len(nn_map)
    in_degree = np.bincount(nn_map, minlength=N)
    top_indices = np.argsort(in_degree)[::-1][:top_n]

    rows, cols = 4, 5
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2, rows * 2.2))
    for i, idx in enumerate(top_indices[:rows * cols]):
        ax = axes[i // cols][i % cols]
        try:
            img = Image.open(image_index[idx]['path']).convert('RGB')
            ax.imshow(img)
        except Exception:
            ax.imshow(np.full((32, 32, 3), 128, dtype=np.uint8))
        ax.set_title(f'hub={in_degree[idx]}\n{image_index[idx]["class"]}', fontsize=8)
        ax.axis('off')

    for i in range(len(top_indices), rows * cols):
        axes[i // cols][i % cols].axis('off')

    fig.suptitle(f'{label} — Top-{top_n} Hub Images', fontsize=13)
    plt.tight_layout()
    plt.show()

show_hub_gallery(nn_d, image_index, 'DINOv2')
show_hub_gallery(nn_c, image_index, 'CLIP')

## Cell 8 — UMAP Projections

In [ ]:
try:
    import umap
except ImportError:
    print('[SKIP] umap-learn not installed. Run: pip install umap-learn')
    umap = None

if umap is None or image_index is None:
    print('[SKIP] UMAP section skipped.')
else:
    from collections import Counter
    class_names = sorted(set(entry['class'] for entry in image_index))
    class_to_idx = {c: i for i, c in enumerate(class_names)}
    class_labels = np.array([class_to_idx[entry['class']] for entry in image_index])

    umap_results = {}

    for emb, label in [(emb_d, 'DINOv2'), (emb_c, 'CLIP')]:
        if emb is None:
            print(f'[SKIP] {label} embeddings not available.')
            continue
        print(f'UMAP fitting {label} embeddings (n={len(emb)}, metric=cosine)...')
        print('This may take 30-60 seconds.')
        reducer = umap.UMAP(
            n_neighbors=15,
            min_dist=0.1,
            metric='cosine',
            random_state=42,
            verbose=True
        )
        embedding_2d = reducer.fit_transform(emb)
        umap_results[label] = embedding_2d
        print(f'Done. Shape: {embedding_2d.shape}')

    cmap_class = plt.cm.get_cmap('tab10', len(class_names))

    for label, emb2d in umap_results.items():
        chains_data = chains_d if label == 'DINOv2' else chains_c
        nn_map = nn_d if label == 'DINOv2' else nn_c
        N = len(emb2d)

        tau_vals = np.array([chains_data['chains'][str(i)]['transient_length'] for i in range(N)])
        hub_vals = np.bincount(nn_map, minlength=N).astype(float)

        cycle_node_set = set()
        for i in range(N):
            for cn in chains_data['chains'][str(i)].get('cycle_nodes', []):
                cycle_node_set.add(cn)
        is_cycle = np.array([i in cycle_node_set for i in range(N)])

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        fig.suptitle(f'{label} UMAP Projections', fontsize=14)

        # 1. By class
        sc = axes[0].scatter(emb2d[:, 0], emb2d[:, 1], c=class_labels,
                             cmap='tab10', s=8, alpha=0.7, vmin=0, vmax=len(class_names) - 1)
        axes[0].scatter(emb2d[is_cycle, 0], emb2d[is_cycle, 1],
                        marker='*', s=80, c='gold', zorder=5, label='Cycle nodes')
        axes[0].set_title('Colored by Class')
        axes[0].legend(fontsize=8)
        for cn in range(len(class_names)):
            mask = class_labels == cn
            cx, cy = emb2d[mask, 0].mean(), emb2d[mask, 1].mean()
            axes[0].annotate(class_names[cn], (cx, cy), fontsize=7, ha='center', alpha=0.8)

        # 2. By transient length
        sc2 = axes[1].scatter(emb2d[:, 0], emb2d[:, 1], c=tau_vals,
                              cmap='viridis', s=8, alpha=0.7)
        axes[1].scatter(emb2d[is_cycle, 0], emb2d[is_cycle, 1],
                        marker='*', s=80, c='gold', zorder=5)
        axes[1].set_title('Colored by τ (transient length)')
        plt.colorbar(sc2, ax=axes[1], label='τ')

        # 3. By hub score
        sc3 = axes[2].scatter(emb2d[:, 0], emb2d[:, 1], c=hub_vals,
                              cmap='hot', s=8, alpha=0.7)
        axes[2].scatter(emb2d[is_cycle, 0], emb2d[is_cycle, 1],
                        marker='*', s=80, c='cyan', zorder=5)
        axes[2].set_title('Colored by Hub Score')
        plt.colorbar(sc3, ax=axes[2], label='in-degree')

        plt.tight_layout()
        plt.show()

## Cell 9 — Cross-Encoder Comparison

In [ ]:
if comparison is None or chains_d is None or chains_c is None:
    print('[SKIP] Comparison data not available. Run: python main.py analyze --compare')
else:
    from scipy.stats import pearsonr

    N = comparison['n_images']

    # ── 1. NN agreement per class ────────────────────────────────────────────
    agree_by_class = comparison['nn_agreement_by_class']
    classes = sorted(agree_by_class.keys())
    rates = [agree_by_class[c] * 100 for c in classes]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].bar(classes, rates, color=sns.color_palette('husl', len(classes)))
    axes[0].axhline(comparison['nn_agreement_rate'] * 100, color='white', linestyle='--', alpha=0.7, label=f'Overall: {comparison["nn_agreement_rate"]*100:.1f}%')
    axes[0].set_xlabel('Class')
    axes[0].set_ylabel('NN Agreement Rate (%)')
    axes[0].set_title('NN Agreement Rate by Class\n(DINOv2 vs CLIP)')
    axes[0].legend()
    axes[0].set_ylim(0, 100)

    # ── 2. τ scatter ─────────────────────────────────────────────────────────
    tau_d = np.array([chains_d['chains'][str(i)]['transient_length'] for i in range(N)])
    tau_c = np.array([chains_c['chains'][str(i)]['transient_length'] for i in range(N)])

    axes[1].scatter(tau_d, tau_c, alpha=0.3, s=12, color='#4a6fa5')
    lim = max(tau_d.max(), tau_c.max()) + 1
    axes[1].plot([0, lim], [0, lim], 'w--', alpha=0.5, label='Identity')
    r, p = pearsonr(tau_d, tau_c)
    axes[1].set_xlabel('τ DINOv2')
    axes[1].set_ylabel('τ CLIP')
    axes[1].set_title(f'Transient Length Correlation\nPearson r={r:.3f} (p={p:.2e})')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    # Summary
    print(f"NN Agreement Rate: {comparison['nn_agreement_rate']*100:.1f}%")
    print(f"Pearson r (τ_DINOv2 vs τ_CLIP): {comparison['tau_pearson_r']:.4f}")
    print(f"Cycle Node Jaccard: {comparison['cycle_node_jaccard']:.4f}")
    print(f"Chain Co-convergence Rate: {comparison['chain_co_convergence_rate']*100:.1f}%")
    print(f"Cycle nodes (DINOv2): {comparison['cycle_nodes_dinov2_count']}")
    print(f"Cycle nodes (CLIP): {comparison['cycle_nodes_clip_count']}")
    print(f"Cycle nodes (intersection): {comparison['cycle_nodes_intersection_count']}")

## Cell 10 — Observations

**Convergence speed:** Most chains converge in ___ steps (avg τ = ___). 
Compared to DINOv2, CLIP chains converge ___ (faster/slower/similar).

**Hub structure:** The top hubs in DINOv2 are visually ___. 
In CLIP they are ___. This suggests the two encoders organize the space differently in ___.

**Fixed points vs longer cycles:** ___% of chains end in fixed points (ℓ=1).
The few longer cycles (ℓ≥2) tend to involve images that ___.

**DINOv2 vs CLIP agreement:** The two encoders agree on the nearest neighbor ___% of the time.
Disagreements cluster in class ___ most heavily, possibly because ___.

**Hypotheses for follow-up:**
1. 
2. 